In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# Residual Connections — Section 9.1

**ResNet** introduces **residual blocks** with skip (shortcut) connections:
$$y = F(x) + x$$

Instead of learning the full transformation directly, the network learns the **residual** $F(x)$ that must be *added* to the input.

**Why it helps with gradients:** During backpropagation:
$$\frac{\partial y}{\partial x} = \frac{\partial F(x)}{\partial x} + 1$$

The `+1` from the identity shortcut ensures gradients can **always flow back** through the network, even when $\partial F / \partial x \approx 0$. This prevents the vanishing gradient problem and enables training networks with 50, 101, or even 152 layers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def compute_gradients(depth, w, residual):
    """Simulate gradient magnitude at each layer.

    Plain: ∂x_l/∂x_{l+1} = w  →  gradient multiplies by w per layer
    Residual: ∂y/∂x = ∂F/∂x + 1 ≈ w + 1  →  gradient multiplies by (1+w)
    """
    grads = np.zeros(depth + 1)
    grads[depth] = 1.0  # output gradient = 1
    for l in range(depth - 1, -1, -1):
        factor = (1 + w) if residual else w
        grads[l] = grads[l + 1] * factor
    return grads


def draw_residual(depth=20, w=0.9, show_plain=True, show_residual=True):
    layers = np.arange(depth + 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for ax in axes:
        ax.set_xlabel('Layer (from output ← input)')
        ax.set_xlim(depth, 0)  # reversed: output on left, input on right

    if show_plain:
        gp = compute_gradients(depth, w, False)
        axes[0].semilogy(layers, np.maximum(gp, 1e-20), color='#dc2626', lw=2,
                         label='Plain network', marker='o', ms=4)
        axes[1].plot(layers, gp / gp[depth], color='#dc2626', lw=2,
                     label='Plain network', marker='o', ms=4)

    if show_residual:
        gr = compute_gradients(depth, w, True)
        axes[0].semilogy(layers, np.maximum(gr, 1e-20), color='#059669', lw=2,
                         label='Residual network', marker='s', ms=4)
        axes[1].plot(layers, gr / gr[depth], color='#059669', lw=2,
                     label='Residual network', marker='s', ms=4)

    axes[0].axhline(1e-7, color='#dc2626', lw=1, ls='--', alpha=0.5)
    axes[0].set_title('Gradient magnitude (log scale)', fontsize=11)
    axes[0].set_ylabel('Gradient magnitude')
    axes[0].grid(True, which='both', alpha=0.3)
    axes[0].legend(fontsize=9)

    axes[1].axhline(0.01, color='#dc2626', lw=1, ls='--', alpha=0.5, label='1% threshold')
    axes[1].set_title('Fraction of output gradient reaching each layer', fontsize=11)
    axes[1].set_ylabel('Relative gradient')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(fontsize=9)

    plt.suptitle(f'Depth = {depth} layers,  weight scale w = {w:.2f}',
                 fontsize=10, y=0.0)
    plt.tight_layout()
    plt.show()

    print(f'\nDepth={depth}, w={w:.2f}')
    if show_plain:
        gp = compute_gradients(depth, w, False)
        print(f'  Plain:    gradient at layer 0 = {gp[0]:.3e}  (w^{depth} = {w**depth:.3e})')
    if show_residual:
        gr = compute_gradients(depth, w, True)
        print(f'  Residual: gradient at layer 0 = {gr[0]:.3e}  ((1+w)^{depth} = {(1+w)**depth:.3e})')
    print()
    print('Formula: y = F(x) + x   →   ∂y/∂x = ∂F/∂x + 1')
    print('The "+1" from the skip connection ensures gradients cannot vanish completely.')


widgets.interact(
    draw_residual,
    depth=widgets.IntSlider(min=4, max=30, step=2, value=20, description='Depth', continuous_update=False),
    w=widgets.FloatSlider(min=0.5, max=1.2, step=0.05, value=0.9, description='Weight scale w', continuous_update=False),
    show_plain=widgets.Checkbox(value=True, description='Show plain'),
    show_residual=widgets.Checkbox(value=True, description='Show residual'),
)